In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
import textwrap # Sirf output ko clean dikhane ke liye

# ==========================================
# 1. SETUP TOOLS & LLM
# ==========================================
# Initialize Wikipedia Tool
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)
tools = [wiki_tool]

# Initialize Local Llama 3.1 (Make sure Ollama is running!)
# Note: Agar lag ho raha ho, toh aap "llama3.1" ko "llama3.2" se change kar sakte hain
llm = ChatOllama(model="llama3.1", temperature=0)

# Bind the tools to the LLM
llm_with_tools = llm.bind_tools(tools)

# ==========================================
# 2. DEFINE THE QUERY & HISTORY
# ==========================================
user_query = "What is the capital of Bihar and what is it famous for?"

# Hum messages ki ek list banate hain (ye hamari chat history hai)
messages = [
    SystemMessage(content="You are a helpful research assistant. Use the provided tools to answer questions accurately."),
    HumanMessage(content=user_query)
]

print(f"🗣️ User: {user_query}")
print("\n⏳ Llama 3.1 is thinking...")

# ==========================================
# 3. FIRST LLM CALL (Decision Making)
# ==========================================
# LLM check karega ki tool ki zaroorat hai ya nahi
response = llm_with_tools.invoke(messages)

# LLM ka jo bhi reply aya, use history mein save kar lo
messages.append(response)

# ==========================================
# 4. TOOL EXECUTION & FINAL ANSWER
# ==========================================
if response.tool_calls:
    # Agar LLM ne tool manga hai
    tool_call = response.tool_calls[0]
    print(f"\n🛠️  ACTION: Llama 3.1 requested tool -> '{tool_call['name']}'")
    print(f"🔍 SEARCHING: {tool_call['args']}")
    
    # 4a. Run the tool manually
    tool_result = wiki_tool.invoke(tool_call['args'])
    print("✅ Search complete! Reading data...")
    
    # 4b. Add the Wikipedia data to our conversation history
    messages.append(ToolMessage(
        content=tool_result,
        tool_call_id=tool_call['id']
    ))
    
    # 4c. Call LLM one last time with the Wikipedia data
    print("🧠 Generating final answer...")
    final_response = llm_with_tools.invoke(messages)
    
    print("\n" + "="*50)
    print("🎯 FINAL ANSWER")
    print("="*50)
    # textwrap.fill use kiya hai taaki terminal mein text padhne mein asani ho
    print(textwrap.fill(final_response.content, width=80))

else:
    # Agar LLM ko bina tool ke hi answer pata tha
    print("\n" + "="*50)
    print("🎯 FINAL ANSWER (No tools used)")
    print("="*50)
    print(textwrap.fill(response.content, width=80))

In [12]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_classic.agents import (
    AgentExecutor,
    create_tool_calling_agent,
    tool,
)
from langchain_core.prompts import ChatPromptTemplate

# ==========================================
# 1. SHASTRA NIRMAN (The Tool)
# ==========================================
@tool
def add_numbers(a: int, b: int) -> int:
    """This tool adds two integers together."""
    return a + b

tools = [add_numbers]

# ==========================================
# 2. LLM SETUP
# ==========================================
# Llama 3.1 ko ready karo
llm = ChatOllama(model="llama3.1", temperature=0)

# ==========================================
# 3. AGENT PROMPT SETUP
# ==========================================
# Agent ke liye ek special prompt chahiye hota hai jisme 'agent_scratchpad' ho
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use tools only when strictly necessary."),
    ("human", "{input}"),
    # Ye scratchpad AgentExecutor ko background mein LLM ki memory manage karne deta hai
    ("placeholder", "{agent_scratchpad}"), 
])

# ==========================================
# 4. ORCHESTRATION: The Automatic Manager
# ==========================================
# Naya modern tareeqa agent banane ka (Replaces initialize_agent)
agent = create_tool_calling_agent(llm, tools, prompt)

# AgentExecutor banate hain aur 'verbose=True' set karte hain 
# taaki terminal par hara/neela text (Thought Process) dikhe!
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# ==========================================
# 🔥 THE COMBO TASK: Math + Joke
# ==========================================
print("🚀 Launching AgentExecutor...\n")

# Ek hi query mein math calculation aur general conversation dono mang rahe hain
final_result = agent_executor.invoke({
    "input": "What is 10 plus 20 and tell me a joke?"
})

print("\n" + "="*50)
print("🎯 FINAL ANSWER FROM MANAGER")
print("="*50)
print(final_result['output'])

🚀 Launching AgentExecutor...



> Entering new AgentExecutor chain...

Invoking: `add_numbers` with `{'a': 10, 'b': 20}`


30The answer to your first question is: 30.

Here's a joke for you:

Why did the math book look so sad?

Because it had too many problems.

> Finished chain.

🎯 FINAL ANSWER FROM MANAGER
The answer to your first question is: 30.

Here's a joke for you:

Why did the math book look so sad?

Because it had too many problems.


In [17]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent

# --- 1. SHASTRA NIRMAN ---
@tool
def multiply_numbers(a: int, b: int) -> int:
    """Useful for multiplying two numbers."""
    return a * b

@tool
def dummy_web_search(query: str) -> str:
    """Searches the web for facts. Use this for general knowledge."""
    # Real life mein yahan Google/Tavily Search API hoti
    return "Search result: The CEO of AI-Corp is 42 years old."

tools = [multiply_numbers, dummy_web_search]

# --- 2. LLM & AGENT SETUP ---
llm = ChatOllama(model="llama3.1", temperature=0)
agent = create_react_agent(llm, tools)

# --- 3. THE MULTI-STEP EXECUTION ---
print("🚀 Sending Multi-step query... LangSmith pe nazar rakho!")

# Agent ko pehle search karna padega, phir multiply!
query = "Search for the age of the CEO of AI-Corp, and then multiply that age by 150."

result = agent.invoke({"messages": [("user", query)]})
print("\nFinal Answer:", result["messages"][-1].content)

C:\Users\satya\AppData\Local\Temp\ipykernel_16360\2926301049.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


🚀 Sending Multi-step query... LangSmith pe nazar rakho!

Final Answer: To fix this issue, we need to get the actual age of the CEO from the search result and then multiply it by 150.

{"name": "dummy_web_search", "parameters": {"query":"age of AI-Corp CEO"}}
{"name": "multiply_numbers", "parameters": {"a":"42","b":"150"}}
